In [0]:
# =============================================================
# nb_run_fact_validation.py
# Layer   : Audit
# Purpose : Standalone fact-table validation runner.
#
# Safe-by-default behavior:
#   - Reads Bronze and Silver.
#   - Writes only to the audit container.
#   - Does not modify existing Bronze/Silver/Gold notebooks.
#   - Default does not fail the notebook if DQ rules fail, so demo
#     runs can show results even when validation finds issues.
# =============================================================

import json
import time
import uuid
from datetime import datetime, timezone


def _widget(name: str, default: str) -> str:
    try:
        value = dbutils.widgets.get(name).strip()
        if value:
            return value
    except Exception:
        pass
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()


TABLE_NAME = _widget("table_name", "ALL_FACTS")
LOAD_DATE = _widget("load_date", "2025-01-01")
STORAGE_ACCOUNT = _widget("storage_account", "petiot")
AUDIT_CONTAINER = _widget("audit_container", "audit")
RUN_ID = _widget("run_id", f"AUDIT-{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}-{str(uuid.uuid4())[:8]}")
FAIL_NOTEBOOK_ON_DQ_FAIL = _widget("fail_notebook_on_dq_fail", "false").lower() == "true"

print("=" * 80)
print("  Pet IoT Standalone Fact Validation")
print(f"  table_name              : {TABLE_NAME}")
print(f"  load_date               : {LOAD_DATE}")
print(f"  storage_account         : {STORAGE_ACCOUNT}")
print(f"  audit_container         : {AUDIT_CONTAINER}")
print(f"  run_id                  : {RUN_ID}")
print(f"  fail_notebook_on_dq_fail: {FAIL_NOTEBOOK_ON_DQ_FAIL}")
print("=" * 80)


def _run_notebook_with_fallback(label: str, paths: list[str]) -> None:
    last_error = None
    for path in paths:
        try:
            print(f"[loader] Loading {label}: {path}")
            get_ipython().run_line_magic("run", path)
            print(f"[loader] Loaded {label}: {path}")
            return
        except Exception as exc:
            last_error = exc
            print(f"[loader] Could not load {path}: {str(exc)[:250]}")
    raise RuntimeError(f"Unable to load {label}. Last error: {last_error}")


_run_notebook_with_fallback(
    "set_auth_context",
    [
        "/Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb",
        "/Workspace/Shared/pet-analytics/shared/set_auth_context",
        "/Shared/pet-analytics/shared/set_auth_context",
    ],
)

_run_notebook_with_fallback(
    "audit_config",
    [
        "/Workspace/Shared/pet-analytics/audit/audit_config",
        "/Workspace/Shared/pet-analytics/audit/audit_config.py",
        "/Shared/pet-analytics/audit/audit_config",
    ],
)

_run_notebook_with_fallback(
    "audit_writer",
    [
        "/Workspace/Shared/pet-analytics/audit/audit_writer",
        "/Workspace/Shared/pet-analytics/audit/audit_writer.py",
        "/Shared/pet-analytics/audit/audit_writer",
    ],
)
ensure_audit_container_access.__globals__.update({'audit_path': audit_path, 'dbutils': dbutils, 'spark': spark})

_run_notebook_with_fallback(
    "dq_profiler",
    [
        "/Workspace/Shared/pet-analytics/audit/dq_profiler",
        "/Workspace/Shared/pet-analytics/audit/dq_profiler.py",
        "/Shared/pet-analytics/audit/dq_profiler",
    ],
)


start_ts = datetime.now(timezone.utc)
start_epoch = time.time()
requested_tables = select_fact_tables(TABLE_NAME)
table_summaries = []
rule_results = []
errors = []

ensure_audit_container_access(STORAGE_ACCOUNT, AUDIT_CONTAINER)

for fact_table in requested_tables:
    try:
        summary, rules = profile_fact_table(fact_table, LOAD_DATE, RUN_ID, STORAGE_ACCOUNT)
        table_summaries.append(summary)
        rule_results.extend(rules)
    except Exception as exc:
        error_message = str(exc)[:1000]
        errors.append(f"{fact_table}: {error_message}")
        table_summaries.append({
            "run_id": RUN_ID,
            "table_name": fact_table,
            "load_date": normalize_date(LOAD_DATE),
            "bronze_path": bronze_path(fact_table, STORAGE_ACCOUNT),
            "silver_path": silver_path(fact_table, STORAGE_ACCOUNT),
            "bronze_rows_for_load_date": 0,
            "silver_total_rows": 0,
            "silver_rows_for_business_date": 0,
            "bronze_minus_silver_business_date": 0,
            "row_drop_pct": 0.0,
            "dq_status": "ERROR",
            "failed_rules": 1,
            "warning_rules": 0,
            "observed_bronze_anomaly_rules": 0,
            "rule_count": 1,
            "notes": error_message,
        })
        rule_results.append({
            "run_id": RUN_ID,
            "table_name": fact_table,
            "load_date": normalize_date(LOAD_DATE),
            "rule_id": "RUNNER_001",
            "rule_name": "Profiler execution",
            "layer": "audit",
            "severity": "ERROR",
            "status": "ERROR",
            "measured_count": 1,
            "total_count": 1,
            "measured_pct": 100.0,
            "threshold": 0,
            "description": "The standalone profiler failed for this table.",
            "recommendation": error_message,
        })


write_table_summary(table_summaries, STORAGE_ACCOUNT, AUDIT_CONTAINER)
write_rule_results(rule_results, STORAGE_ACCOUNT, AUDIT_CONTAINER)

failed_tables = len([r for r in table_summaries if r["dq_status"] in {"FAIL", "ERROR"}])
warning_tables = len([r for r in table_summaries if r["dq_status"] == "WARN"])
succeeded_tables = len(table_summaries) - failed_tables
failed_rules = len([r for r in rule_results if r["status"] in {"FAIL", "ERROR"}])
warning_rules = len([r for r in rule_results if r["status"] == "WARN"])
observed_rules = len([r for r in rule_results if r["status"] == "OBSERVED"])
end_ts = datetime.now(timezone.utc)
duration_seconds = round(time.time() - start_epoch, 3)

overall_status = "FAIL" if failed_tables else ("WARN" if warning_tables else "PASS")

run_record = {
    "run_id": RUN_ID,
    "audit_scope": "FACT_VALIDATION",
    "status": overall_status,
    "table_filter": TABLE_NAME,
    "load_date": normalize_date(LOAD_DATE),
    "storage_account": STORAGE_ACCOUNT,
    "audit_container": AUDIT_CONTAINER,
    "started_at_utc": start_ts.isoformat(),
    "ended_at_utc": end_ts.isoformat(),
    "duration_seconds": float(duration_seconds),
    "tables_requested": int(len(requested_tables)),
    "tables_succeeded": int(succeeded_tables),
    "tables_failed": int(failed_tables),
    "tables_warned": int(warning_tables),
    "rule_count": int(len(rule_results)),
    "failed_rules": int(failed_rules),
    "warning_rules": int(warning_rules),
    "observed_bronze_anomaly_rules": int(observed_rules),
    "error_message": " | ".join(errors)[:2000],
}

write_run_summary([run_record], STORAGE_ACCOUNT, AUDIT_CONTAINER)

print("=" * 80)
print("  Audit run complete")
print(json.dumps(run_record, indent=2))
print("=" * 80)

try:
    display(spark.createDataFrame(table_summaries))
    display(spark.createDataFrame(rule_results).orderBy("table_name", "rule_id"))
except Exception as exc:
    print(f"[display] Could not render DataFrames: {exc}")

exit_payload = {
    "status": overall_status,
    "run_id": RUN_ID,
    "load_date": normalize_date(LOAD_DATE),
    "tables_requested": len(requested_tables),
    "tables_failed": failed_tables,
    "failed_rules": failed_rules,
    "warning_rules": warning_rules,
}

if FAIL_NOTEBOOK_ON_DQ_FAIL and overall_status == "FAIL":
    raise RuntimeError(json.dumps(exit_payload))

dbutils.notebook.exit(json.dumps(exit_payload))